# C2: the generation loop

**Goal:** our model -> C1 sampler -> append token -> repeat.

```
run the current prefix
choose from its LAST logits row
append the new token
stop on newly generated EOS or max_new_tokens
```

**Spec:** `generate` returns prompt + new IDs, including generated EOS, with unchanged dtype/device and no mutation of the input. Greedy is default; C1 options enable sampling. Zero new tokens returns a copy without forward. Oversized prompt + requested budget raises before generation; no silent truncation.

This is single-request, no-cache generation. C3 will reuse old K/V instead of recomputing the entire prefix.

Run top to bottom on a **fresh Colab T4 runtime**. Two FP32 model copies need about 8.8 GB of weights plus temporary memory. Close older model notebook kernels; save results before deleting an old runtime.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


In [ ]:
import importlib, gc
import torch
import tinyserve.sampling, tinyserve.generation
for module in (tinyserve.sampling, tinyserve.generation):
    importlib.reload(module)
from tinyserve.generation import generate
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.basics import load_model, encode
from tinyserve.envinfo import collect_env, save_result

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())


## 1. Test first

Run B8/C1/C2 tests. Scripted models expose the prefixes processed at every step and verify immediate EOS, EOS lists, prompt EOS, exact budgets, context errors, zero-token requests and invalid inputs.

Small HF models verify greedy output against `HF.generate(use_cache=False)`, including forced EOS. Seeded sampling is checked against the C1 sampler's manual draw sequence.

A failed test stops the notebook. HF and CUDA checks should be available here.

In [ ]:
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b8_model.py", "tests/test_c1_sampling.py",
                           "tests/test_c2_generation.py", "-v", "-ra"], capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. Load our model and HF's reference in FP32

The trained weights are the same; our generation loop calls our model, not HF. Keep FP32 for token parity. If loading kills the kernel, use a fresh whole Colab runtime rather than repeatedly restarting one notebook kernel.

In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
hf_model.set_attn_implementation("eager")
our_model = LlamaForCausalLM.from_huggingface(hf_model)
print("our model:", type(our_model).__name__, "dtype:", next(our_model.parameters()).dtype)
print("EOS ID:", our_model.config.eos_token_id, "context limit:", our_model.config.max_position_embeddings)


## 3. Raw continuation versus a chat template

Raw text asks for document continuation and can ramble into lists. A chat template marks system/user/assistant turns using the model's expected special tokens.

Use `apply_chat_template(..., tokenize=True)` directly: do not tokenize that output again or add a second BOS. Earlier chat-turn EOS delimiters are prompt content, not a request to stop the new answer.

In [ ]:
raw_ids = encode(tokenizer, "The capital of France is", DEVICE)
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France? Answer in one sentence."},
]
chat_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")[0].to(DEVICE)
print("raw prompt tokens:", raw_ids.numel(), "chat prompt tokens:", chat_ids.numel())
print("chat template with special tokens:\n", tokenizer.decode(chat_ids.tolist(), skip_special_tokens=False))


## 4. Greedy generation and token-exact HF comparison

For each prompt, our loop recomputes the full prefix and selects the last-row argmax. HF also runs without a cache. Assert the entire returned sequence is identical, not just the displayed text.

Output includes prompt plus continuation. Slice off the prompt for the answer alone. The printed stop reason is observed: chat formatting does not guarantee EOS.

In [ ]:
greedy_rows = []
for label, prompt_ids, budget in (("raw", raw_ids, 12), ("chat", chat_ids, 24)):
    generated_ids = generate(our_model, prompt_ids, max_new_tokens=budget)
    with torch.no_grad():
        hf_ids = hf_model.generate(prompt_ids[None], attention_mask=torch.ones_like(prompt_ids)[None],
                                   do_sample=False, max_new_tokens=budget, use_cache=False,
                                   pad_token_id=hf_model.config.eos_token_id)[0]
    assert torch.equal(generated_ids, hf_ids), f"greedy token parity failed for {label}"
    continuation = generated_ids[prompt_ids.numel():]
    stop_reason = "eos" if continuation[-1].item() == our_model.config.eos_token_id else "token_limit"
    answer = tokenizer.decode(continuation.tolist(), skip_special_tokens=True)
    greedy_rows.append({"prompt_type": label, "prompt_tokens": prompt_ids.numel(), "budget": budget,
                        "new_tokens": continuation.numel(), "stop_reason": stop_reason,
                        "token_ids": generated_ids.tolist(), "continuation": answer, "hf_token_equal": True})
    print(f"{label}: {continuation.numel()} new tokens, stop={stop_reason}, HF equal=True, answer={answer!r}")


## 5. Seeded sampled generation

Free HF's model now: only our model is needed for this demo. Use temperature 0.8, top-k 40 and top-p 0.9. Unlike C1's unchanged-row draws, each sampled token changes the next model input.

Recreate the same GPU generator seed and run again to verify repeatability on this runtime. No stochastic token equality with a separate HF implementation is claimed.

In [ ]:
del hf_model
gc.collect()
torch.cuda.empty_cache()
sampled_ids = generate(our_model, chat_ids, max_new_tokens=24, temperature=0.8, top_k=40, top_p=0.9,
                       generator=torch.Generator(device=DEVICE).manual_seed(17))
repeated_ids = generate(our_model, chat_ids, max_new_tokens=24, temperature=0.8, top_k=40, top_p=0.9,
                        generator=torch.Generator(device=DEVICE).manual_seed(17))
assert torch.equal(sampled_ids, repeated_ids)
sampled_continuation = sampled_ids[chat_ids.numel():]
sampled_answer = tokenizer.decode(sampled_continuation.tolist(), skip_special_tokens=True)
print("sampled answer:", repr(sampled_answer))
print("same-seed replay equal:", torch.equal(sampled_ids, repeated_ids))


## 6. Zero budget and strict context errors

Zero new tokens returns a copy of the prompt and does not run the model. An oversized requested budget raises before forward, even if the model might have emitted EOS early. This demonstrates the chosen boundary policy; it does not silently shorten the request.

In [ ]:
zero_ids = generate(our_model, raw_ids, max_new_tokens=0)
assert torch.equal(zero_ids, raw_ids) and zero_ids.data_ptr() != raw_ids.data_ptr()
try:
    generate(our_model, raw_ids, max_new_tokens=our_model.config.max_position_embeddings)
except ValueError as error:
    context_error = str(error)
    print("expected error:", context_error)
else:
    raise AssertionError("oversized request should have raised")


## 7. Save evidence and three interview questions

1. **Why the last logits row?** It predicts the token after the complete current prefix.
2. **EOS versus max_new_tokens?** EOS is a selected end marker; the new-token count is a separate hard bound.
3. **What repeats unnecessarily?** Every old token's computations, including its K/V. C3 will cache them.

Save with outputs and retain the JSON. No speed claims are made here; the range below is across demonstrated generation lengths, not benchmark trials.

**Next: C3 contiguous KV cache**, followed by C4's measured no-cache vs cached speed curve.

In [ ]:
sampled_stop = "eos" if sampled_continuation[-1].item() == our_model.config.eos_token_id else "token_limit"
new_token_counts = [row["new_tokens"] for row in greedy_rows] + [sampled_continuation.numel()]
saved_path = save_result("C2_generation", {
    "dtype": "float32", "use_cache": False,
    "greedy_comparison": greedy_rows,
    "sampled_generation": {"temperature": 0.8, "top_k": 40, "top_p": 0.9, "seed": 17,
                           "token_ids": sampled_ids.tolist(), "continuation": sampled_answer,
                           "new_tokens": sampled_continuation.numel(), "stop_reason": sampled_stop,
                           "same_seed_replay_equal": torch.equal(sampled_ids, repeated_ids)},
    "zero_budget_returns_prompt": torch.equal(zero_ids, raw_ids),
    "oversized_context_error": context_error,
    "new_token_count_range_across_demos": {"min": min(new_token_counts), "max": max(new_token_counts)},
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


In [ ]:
del our_model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
